# Sewer Capacity Analysis — sample_capacity_sp.xlsx
Pure Python stdlib + openpyxl. Prepares and analyses 1,634 pipe segments
across multiple capacity-estimation methods (GIS, IDM, TCEQ, HouCAT/ICM).

In [1]:
import openpyxl, json, math, re
from collections import Counter, defaultdict
from statistics import mean, median, stdev

# ── Load raw data ──────────────────────────────────────────────────────────────
wb = openpyxl.load_workbook('sample_capacity_sp.xlsx', data_only=True)
ws = wb['Sheet1']
rows = list(ws.iter_rows(min_row=1, values_only=True))
HEADERS = [str(h) if h is not None else '' for h in rows[0]]
DATA_RAW = rows[1:]
print(f'Rows: {len(DATA_RAW)}  Cols: {len(HEADERS)}')
print('Headers sample:', HEADERS[:10])

Rows: 1634  Cols: 80
Headers sample: ['UFID', 'BASIN', 'OUTFALL_UFID', 'PIPE_DIAM', 'TOTAL ACRE', 'GIS_SLOPE(%)', 'GIS_MGD', 'GIS_GPAD', 'GIS_GPAD_CLASS', 'MANNING_N']


In [2]:
# ── Helper utilities ───────────────────────────────────────────────────────────
def idx(name):
    return HEADERS.index(name)

def safe_float(v, default=None):
    if v is None or str(v).strip() in ('', 'None', 'nan', 'NULL'):
        return default
    try:
        return float(v)
    except (ValueError, TypeError):
        return default

def safe_str(v):
    if v is None or str(v).strip() in ('None', 'nan', 'NULL', ''):
        return None
    return str(v).strip()

# Q-class ordering (for sorting / colour mapping)
QCLASS_ORDER = {'LT4Q': 1, '4Q to 6Q': 2, '6Q to 9Q': 3, '9Q to 12Q': 4, '12Q to 15Q': 5, 'GTE 15Q': 6}
QCLASS_LABEL = {1: 'LT4Q', 2: '4Q–6Q', 3: '6Q–9Q', 4: '9Q–12Q', 5: '12Q–15Q', 6: '≥15Q'}

def qclass_no(label):
    return QCLASS_ORDER.get(str(label).strip(), 0) if label else 0

# Surcharge status: >1 means surcharging (d/D > 1)
def is_surcharged(v):
    f = safe_float(v)
    return f is not None and f > 1.0

# Capacity status from depth ratio
def depth_status(v):
    f = safe_float(v)
    if f is None: return 'Unknown'
    if f > 1.0:  return 'Surcharged'
    if f > 0.85: return 'Critical'
    if f > 0.75: return 'At-Risk'
    return 'Adequate'

print('Helpers ready.')

Helpers ready.


In [3]:

# ── Build clean record list ────────────────────────────────────────────────────
import math

def fin(v):
    return v is not None and math.isfinite(float(v)) if v is not None else False

records = []
for r in DATA_RAW:
    basin = safe_str(r[idx('BASIN')]) or 'Unknown'

    diam  = safe_float(r[idx('PIPE_DIAM')])
    slope_gis  = safe_float(r[idx('GIS_SLOPE(%)')])
    slope_idm  = safe_float(r[idx('IDM_SLOPE(%)')])
    slope_tceq = safe_float(r[idx('TCEQ_SLOPE(%)')])

    gis_mgd   = safe_float(r[idx('GIS_MGD')])
    idm_mgd   = safe_float(r[idx('IDM_MGD')])
    tceq_mgd  = safe_float(r[idx('TCEQ_MGD')])
    dwf_mgd   = safe_float(r[idx('ESU_BASED_DWF_MGD')])

    # Q_RATIO in this dataset = capacity / flow (higher is better; >1 means adequate)
    gis_qratio  = safe_float(r[idx('GIS_Q_RATIO')])
    idm_qratio  = safe_float(r[idx('IDM_Q_RATIO')])
    tceq_qratio = safe_float(r[idx('TCEQ_Q_RATIO')])
    mod_qratio  = safe_float(r[idx('MODEL_FLOW_Q_RATIO')])

    gis_qclass  = safe_str(r[idx('GIS_Q_CLASS')])
    idm_qclass  = safe_str(r[idx('IDM_Q_CLASS')])
    tceq_qclass = safe_str(r[idx('TCEQ_Q_CLASS')])
    mod_qclass  = safe_str(r[idx('MODEL_FLOW_Q_CLASS')])

    gpad         = safe_float(r[idx('GPAD')])
    gis_gpad     = safe_float(r[idx('GIS_GPAD')])
    model_type   = safe_str(r[idx('MODEL_TYPE')]) or 'Unknown'

    # SURCH_STAT = depth/diameter ratio from model; >1 = surcharging
    surch_stat   = safe_float(r[idx('SURCH_STAT')])
    icm_surch    = safe_float(r[idx('ICM_SURCH_STAT')])

    peak_flow_2y6h  = safe_float(r[idx('MAX_2Y6H_FLOW_FINAL')])
    peak_depth_2y6h = safe_float(r[idx('MAX_2Y6H_DEPTH_FINAL')])

    # q/Q column: standard flow/full-capacity ratio; >1 = surcharging
    q_over_Q  = safe_float(r[idx('q/Q')])
    gis_pct   = safe_float(r[idx('GIS_PCT')])
    tceq_pct  = safe_float(r[idx('TCEQ_PCT')])
    idm_pct   = safe_float(r[idx('IDM_PCT')])
    max_pct   = safe_float(r[idx('MAX_2Y6H_PCT')])

    total_acre = safe_float(r[idx('TOTAL_ACRE')])
    imp_acre   = safe_float(r[idx('IMP ACRE')])
    parcel_cnt = safe_float(r[idx('PARCEL_CNT')])

    # Diameter band
    if diam is None:         diam_band = 'Unknown'
    elif diam <= 8:          diam_band = '≤8"'
    elif diam <= 12:         diam_band = '10–12"'
    elif diam <= 18:         diam_band = '15–18"'
    elif diam <= 24:         diam_band = '21–24"'
    elif diam <= 36:         diam_band = '27–36"'
    else:                    diam_band = '>36"'

    # Surcharge: d/D > 1 under 2yr/6hr storm
    surcharged = (surch_stat is not None and math.isfinite(surch_stat) and surch_stat > 1.0)
    icm_surcharged = (icm_surch is not None and math.isfinite(icm_surch) and icm_surch > 1.0)

    # Depth-ratio status (SURCH_STAT = d/D from model)
    if surch_stat is None or not math.isfinite(surch_stat):
        d_status = 'Unknown'
    elif surch_stat > 1.0:  d_status = 'Surcharged'
    elif surch_stat > 0.85: d_status = 'Critical'
    elif surch_stat > 0.75: d_status = 'At-Risk'
    else:                   d_status = 'Adequate'

    # q/Q based deficiency (standard hydraulics: flow/capacity > 1 = overloaded)
    qQ_deficient = (q_over_Q is not None and math.isfinite(q_over_Q) and q_over_Q > 1.0)

    # Worst capacity-to-flow ratio (lower = less capacity headroom; LT4Q = tightest)
    # MODEL is most meaningful; fall back to GIS
    primary_qratio = mod_qratio if (mod_qratio is not None and math.isfinite(mod_qratio)) else \
                     gis_qratio if (gis_qratio is not None and math.isfinite(gis_qratio)) else None

    records.append({
        'ufid': r[idx('UFID')],
        'basin': basin,
        'diam': diam,
        'diam_band': diam_band,
        'model_type': model_type,
        'total_acre': total_acre,
        'imp_acre': imp_acre,
        'parcel_cnt': parcel_cnt,
        'slope_gis': slope_gis,
        'slope_idm': slope_idm,
        'slope_tceq': slope_tceq,
        'gis_mgd': gis_mgd, 'idm_mgd': idm_mgd, 'tceq_mgd': tceq_mgd,
        'dwf_mgd': dwf_mgd,
        'gis_qratio': gis_qratio, 'idm_qratio': idm_qratio,
        'tceq_qratio': tceq_qratio, 'mod_qratio': mod_qratio,
        'primary_qratio': primary_qratio,
        'gis_qclass': gis_qclass, 'idm_qclass': idm_qclass,
        'tceq_qclass': tceq_qclass, 'mod_qclass': mod_qclass,
        'gpad': gpad, 'gis_gpad': gis_gpad,
        'surch_stat': surch_stat,
        'icm_surch': icm_surch,
        'surcharged': surcharged,
        'icm_surcharged': icm_surcharged,
        'depth_status': d_status,
        'qQ_deficient': qQ_deficient,
        'peak_flow_2y6h': peak_flow_2y6h,
        'peak_depth_2y6h': peak_depth_2y6h,
        'q_over_Q': q_over_Q,
        'gis_pct': gis_pct, 'tceq_pct': tceq_pct, 'idm_pct': idm_pct,
        'max_pct': max_pct,
    })

print(f'Clean records: {len(records)}')
print('Fields:', list(records[0].keys())[:12], '...')


Clean records: 1634
Fields: ['ufid', 'basin', 'diam', 'diam_band', 'model_type', 'total_acre', 'imp_acre', 'parcel_cnt', 'slope_gis', 'slope_idm', 'slope_tceq', 'gis_mgd'] ...


In [4]:

# ── Summary KPIs ───────────────────────────────────────────────────────────────
n = len(records)

# Primary surcharge metric: SURCH_STAT d/D > 1 under 2yr/6hr storm
n_surcharged    = sum(1 for r in records if r['surcharged'])
n_icm_surcharged= sum(1 for r in records if r['icm_surcharged'])
n_qQ_deficient  = sum(1 for r in records if r['qQ_deficient'])

# Depth status distribution
n_critical  = sum(1 for r in records if r['depth_status'] == 'Critical')
n_at_risk   = sum(1 for r in records if r['depth_status'] == 'At-Risk')
n_adequate  = sum(1 for r in records if r['depth_status'] == 'Adequate')
n_unknown   = sum(1 for r in records if r['depth_status'] == 'Unknown')

n_houcat = sum(1 for r in records if r['model_type'] == 'HouCAT')
n_icm    = sum(1 for r in records if r['model_type'] == 'ICM')

# MODEL_FLOW_Q_CLASS distribution
model_lt4q  = sum(1 for r in records if r['mod_qclass'] == 'LT4Q')
model_gte15 = sum(1 for r in records if r['mod_qclass'] == 'GTE 15Q')

# q/Q statistics (final column: flow/capacity; > 1 = overloaded)
qQ_vals = [r['q_over_Q'] for r in records
           if r['q_over_Q'] is not None and math.isfinite(r['q_over_Q'])]
avg_qQ  = round(mean(qQ_vals), 3) if qQ_vals else None
max_qQ  = round(max(qQ_vals), 3) if qQ_vals else None

# GPAD statistics
gpad_vals = [r['gpad'] for r in records
             if r['gpad'] is not None and r['gpad'] > 0 and math.isfinite(r['gpad'])]
avg_gpad = round(mean(gpad_vals)) if gpad_vals else None

print(f'Total segments           : {n}')
print(f'Surcharged (d/D>1)       : {n_surcharged} ({100*n_surcharged/n:.1f}%)')
print(f'ICM surcharged           : {n_icm_surcharged}')
print(f'q/Q > 1 deficient        : {n_qQ_deficient} ({100*n_qQ_deficient/n:.1f}%)')
print(f'Critical (d/D 0.85-1.0)  : {n_critical} ({100*n_critical/n:.1f}%)')
print(f'At-Risk (d/D 0.75-0.85)  : {n_at_risk} ({100*n_at_risk/n:.1f}%)')
print(f'Adequate (d/D <0.75)     : {n_adequate} ({100*n_adequate/n:.1f}%)')
print(f'Model LT4Q / GTE15Q      : {model_lt4q} / {model_gte15}')
print(f'HouCAT / ICM             : {n_houcat} / {n_icm}')
print(f'Avg q/Q                  : {avg_qQ}  Max: {max_qQ}')
print(f'Avg GPAD                 : {avg_gpad}')


Total segments           : 1634
Surcharged (d/D>1)       : 105 (6.4%)
ICM surcharged           : 95
q/Q > 1 deficient        : 1401 (85.7%)
Critical (d/D 0.85-1.0)  : 129 (7.9%)
At-Risk (d/D 0.75-0.85)  : 23 (1.4%)
Adequate (d/D <0.75)     : 1353 (82.8%)
Model LT4Q / GTE15Q      : 1387 / 50
HouCAT / ICM             : 1090 / 531
Avg q/Q                  : 27.027  Max: 845.557
Avg GPAD                 : 18313


In [5]:

# ── Basin-level aggregation ────────────────────────────────────────────────────
basin_data = defaultdict(lambda: {
    'count': 0, 'surcharged': 0, 'qQ_def': 0, 'critical': 0, 'at_risk': 0,
    'qratios': [], 'gpads': [], 'acres': [], 'diams': []
})

for r in records:
    b = r['basin']
    d = basin_data[b]
    d['count'] += 1
    if r['surcharged']:     d['surcharged'] += 1
    if r['qQ_deficient']:   d['qQ_def']     += 1
    if r['depth_status'] == 'Critical': d['critical'] += 1
    if r['depth_status'] == 'At-Risk':  d['at_risk']  += 1
    if r['primary_qratio'] is not None: d['qratios'].append(r['primary_qratio'])
    if fin(r['gpad']) and r['gpad'] > 0: d['gpads'].append(r['gpad'])
    if fin(r['total_acre']): d['acres'].append(r['total_acre'])
    if fin(r['diam']): d['diams'].append(r['diam'])

basins_summary = []
for basin, d in sorted(basin_data.items(), key=lambda x: -x[1]['count']):
    cnt = d['count']
    avg_gpad_b = None
    if d['gpads']:
        g = mean(d['gpads'])
        avg_gpad_b = round(g) if math.isfinite(g) else None
    basins_summary.append({
        'basin': basin,
        'count': cnt,
        'surcharged': d['surcharged'],
        'qQ_def': d['qQ_def'],
        'critical': d['critical'],
        'at_risk': d['at_risk'],
        'surcharge_pct': round(100 * d['surcharged'] / cnt, 1),
        'qQ_def_pct': round(100 * d['qQ_def'] / cnt, 1),
        'avg_qratio': round(mean(d['qratios']), 3) if d['qratios'] else None,
        'min_qratio': round(min(d['qratios']), 3) if d['qratios'] else None,
        'avg_gpad': avg_gpad_b,
        'total_acre': round(sum(d['acres']), 1) if d['acres'] else None,
        'avg_diam': round(mean(d['diams']), 1) if d['diams'] else None,
    })

print(f'Basins: {len(basins_summary)}')
for b in basins_summary[:10]:
    print(f"  {b['basin']:8s}  n={b['count']:4d}  surcharge={b['surcharge_pct']:5.1f}%  q/Q_def={b['qQ_def_pct']:5.1f}%  avg_qratio={b['avg_qratio']}")


Basins: 38
  BW230     n= 180  surcharge=  6.1%  q/Q_def= 76.1%  avg_qratio=3.27
  BW232     n= 116  surcharge=  4.3%  q/Q_def= 97.4%  avg_qratio=3.622
  TK205     n=  96  surcharge=  2.1%  q/Q_def= 94.8%  avg_qratio=2.37
  BW245     n=  95  surcharge= 13.7%  q/Q_def= 86.3%  avg_qratio=2.792
  BW240     n=  91  surcharge=  0.0%  q/Q_def= 89.0%  avg_qratio=1.755
  TK221     n=  84  surcharge=  4.8%  q/Q_def= 95.2%  avg_qratio=5.277
  3         n=  81  surcharge=  2.5%  q/Q_def= 77.8%  avg_qratio=2.176
  4         n=  81  surcharge=  0.0%  q/Q_def= 95.1%  avg_qratio=2.582
  6         n=  81  surcharge=  1.2%  q/Q_def= 98.8%  avg_qratio=2.585
  7         n=  81  surcharge=  1.2%  q/Q_def= 97.5%  avg_qratio=5.948


In [6]:
# ── Q-class distributions across all four methods ─────────────────────────────
METHODS = [
    ('GIS',   'gis_qclass'),
    ('IDM',   'idm_qclass'),
    ('TCEQ',  'tceq_qclass'),
    ('Model', 'mod_qclass'),
]
QCLASS_LABELS = ['LT4Q', '4Q to 6Q', '6Q to 9Q', '9Q to 12Q', '12Q to 15Q', 'GTE 15Q']

qclass_by_method = {}
for method_name, field in METHODS:
    counts = Counter()
    for r in records:
        v = r[field]
        if v:
            counts[v] += 1
        else:
            counts['Unknown'] += 1
    qclass_by_method[method_name] = {lbl: counts.get(lbl, 0) for lbl in QCLASS_LABELS}
    qclass_by_method[method_name]['Unknown'] = counts.get('Unknown', 0)

print('Q-class distributions by method:')
for m, dist in qclass_by_method.items():
    print(f'  {m:6s}: {dist}')

Q-class distributions by method:
  GIS   : {'LT4Q': 336, '4Q to 6Q': 130, '6Q to 9Q': 105, '9Q to 12Q': 97, '12Q to 15Q': 60, 'GTE 15Q': 906, 'Unknown': 0}
  IDM   : {'LT4Q': 237, '4Q to 6Q': 123, '6Q to 9Q': 98, '9Q to 12Q': 110, '12Q to 15Q': 61, 'GTE 15Q': 1005, 'Unknown': 0}
  TCEQ  : {'LT4Q': 286, '4Q to 6Q': 117, '6Q to 9Q': 109, '9Q to 12Q': 90, '12Q to 15Q': 67, 'GTE 15Q': 965, 'Unknown': 0}
  Model : {'LT4Q': 1387, '4Q to 6Q': 110, '6Q to 9Q': 33, '9Q to 12Q': 24, '12Q to 15Q': 6, 'GTE 15Q': 50, 'Unknown': 24}


In [7]:

# ── Diameter-band analysis ─────────────────────────────────────────────────────
DIAM_BANDS = ['≤8"', '10–12"', '15–18"', '21–24"', '27–36"', '>36"']

diam_data = defaultdict(lambda: {'count': 0, 'surcharged': 0, 'qQ_def': 0, 'qratios': []})
for r in records:
    b = r['diam_band']
    diam_data[b]['count'] += 1
    if r['surcharged']:   diam_data[b]['surcharged'] += 1
    if r['qQ_deficient']: diam_data[b]['qQ_def']     += 1
    if r['primary_qratio'] is not None:
        diam_data[b]['qratios'].append(r['primary_qratio'])

diam_summary = []
for band in DIAM_BANDS:
    d = diam_data[band]
    cnt = d['count']
    if cnt == 0: continue
    diam_summary.append({
        'band': band,
        'count': cnt,
        'surcharged': d['surcharged'],
        'qQ_def': d['qQ_def'],
        'surcharge_pct': round(100 * d['surcharged'] / cnt, 1),
        'qQ_def_pct': round(100 * d['qQ_def'] / cnt, 1),
        'avg_qratio': round(mean(d['qratios']), 2) if d['qratios'] else None,
        'min_qratio': round(min(d['qratios']), 2) if d['qratios'] else None,
    })

print('Diameter band breakdown:')
for d in diam_summary:
    print(f"  {d['band']:8s}  n={d['count']:4d}  surcharged={d['surcharge_pct']:5.1f}%  q/Q_def={d['qQ_def_pct']:5.1f}%  avg_qr={d['avg_qratio']}")


Diameter band breakdown:
  ≤8"       n=1079  surcharged=  0.7%  q/Q_def= 85.0%  avg_qr=3.54
  10–12"    n= 308  surcharged= 20.5%  q/Q_def= 85.7%  avg_qr=5.81
  15–18"    n= 142  surcharged= 23.2%  q/Q_def= 86.6%  avg_qr=3.02
  21–24"    n=  54  surcharged=  1.9%  q/Q_def= 88.9%  avg_qr=1.42
  27–36"    n=   8  surcharged=  0.0%  q/Q_def=100.0%  avg_qr=1.37
  >36"      n=  43  surcharged=  0.0%  q/Q_def= 95.3%  avg_qr=2.64


In [8]:

# ── GPAD class distribution ────────────────────────────────────────────────────
GPAD_CLASSES = ['<5K', '5K-7.5K', '7.5K-10K', '10K-13.5K', '13.5K-15K', 'GTE 15K']
MOD_GPAD_IDX = idx('MODEL_FLOW_GPAD_CLASS')

gpad_class_counts = Counter()
for r_raw in DATA_RAW:
    v = safe_str(r_raw[MOD_GPAD_IDX])
    gpad_class_counts[v if v else 'Unknown'] += 1

# GPAD histogram from computed GPAD values
gpad_bins = [0, 2500, 5000, 7500, 10000, 12500, 15000, 20000, 30000]
gpad_hist = [0] * (len(gpad_bins) - 1)
for r in records:
    g = r['gpad']
    if g is None or g <= 0 or not math.isfinite(g): continue
    for i in range(len(gpad_bins) - 1):
        if gpad_bins[i] <= g < gpad_bins[i+1]:
            gpad_hist[i] += 1
            break

print('GPAD class distribution:')
for cls in GPAD_CLASSES:
    print(f'  {cls:12s}: {gpad_class_counts.get(cls, 0)}')
print('GPAD histogram bins:', gpad_hist)


GPAD class distribution:
  <5K         : 172
  5K-7.5K     : 73
  7.5K-10K    : 130
  10K-13.5K   : 151
  13.5K-15K   : 283
  GTE 15K     : 800
GPAD histogram bins: [52, 68, 73, 130, 86, 348, 423, 59]


In [9]:

# ── q/Q histogram and scatter data ────────────────────────────────────────────
depth_status_counts = Counter(r['depth_status'] for r in records)

# q/Q histogram (final column: flow/capacity ratio)
qQ_vals = [r['q_over_Q'] for r in records
           if r['q_over_Q'] is not None and math.isfinite(r['q_over_Q'])]
qQ_bins = [0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 5.0, 1000.0]
qQ_labels = ['0–0.1','0.1–0.25','0.25–0.5','0.5–0.75','0.75–1.0','1.0–1.5','1.5–2.0','2.0–5.0','5.0+']
qQ_hist = [0] * len(qQ_labels)
for v in qQ_vals:
    for i in range(len(qQ_bins) - 1):
        if qQ_bins[i] <= v < qQ_bins[i+1]:
            qQ_hist[i] += 1
            break

# Scatter data: diam vs model q/Q ratio coloured by depth status
def scatter_point(r):
    qr = r['primary_qratio']
    return {
        'diam': r['diam'],
        'qr': round(qr, 2) if (qr is not None and math.isfinite(qr)) else None,
        'basin': r['basin'],
        'status': r['depth_status'],
        'surch': r['surch_stat'],
    }

import random; random.seed(42)
surcharged_recs = [r for r in records if r['surcharged']]
critical_recs   = [r for r in records if r['depth_status'] == 'Critical']
other_recs      = [r for r in records if not r['surcharged'] and r['depth_status'] not in ('Critical',)]
scatter_sample  = (
    [scatter_point(r) for r in surcharged_recs] +
    [scatter_point(r) for r in critical_recs] +
    [scatter_point(r) for r in random.sample(other_recs, min(100, len(other_recs)))]
)

print('Depth status:', dict(depth_status_counts))
print('q/Q histogram:', qQ_hist)
print('Scatter sample size:', len(scatter_sample))


Depth status: {'Adequate': 1353, 'Critical': 129, 'Surcharged': 105, 'Unknown': 24, 'At-Risk': 23}
q/Q histogram: [54, 6, 6, 25, 10, 45, 30, 243, 1083]
Scatter sample size: 334


In [10]:
# ── Model-type vs capacity class cross-tab ─────────────────────────────────────
# For each model type, count pipes per MODEL_FLOW_Q_CLASS
mt_qclass = defaultdict(Counter)
for r in records:
    mt = r['model_type']
    qc = r['mod_qclass'] or 'Unknown'
    mt_qclass[mt][qc] += 1

print('Model-type × Q-class cross-tab:')
for mt, counts in mt_qclass.items():
    print(f'  {mt}: {dict(counts)}')

Model-type × Q-class cross-tab:
  HouCAT: {'6Q to 9Q': 17, 'LT4Q': 989, 'GTE 15Q': 31, '4Q to 6Q': 40, '9Q to 12Q': 10, '12Q to 15Q': 3}
  ICM: {'LT4Q': 398, 'Unknown': 11, '4Q to 6Q': 70, '6Q to 9Q': 16, 'GTE 15Q': 19, '9Q to 12Q': 14, '12Q to 15Q': 3}
  Unknown: {'Unknown': 13}


In [11]:

# ── Export dashboard data JSON ─────────────────────────────────────────────────
dashboard_data = {
    'meta': {
        'total_segments': n,
        'n_surcharged': n_surcharged,
        'n_icm_surcharged': n_icm_surcharged,
        'n_qQ_deficient': n_qQ_deficient,
        'n_critical': n_critical,
        'n_at_risk': n_at_risk,
        'n_adequate': n_adequate,
        'n_unknown': n_unknown,
        'surcharge_pct': round(100 * n_surcharged / n, 1),
        'qQ_def_pct': round(100 * n_qQ_deficient / n, 1),
        'critical_pct': round(100 * n_critical / n, 1),
        'at_risk_pct': round(100 * n_at_risk / n, 1),
        'adequate_pct': round(100 * n_adequate / n, 1),
        'n_houcat': n_houcat,
        'n_icm': n_icm,
        'model_lt4q': model_lt4q,
        'model_gte15': model_gte15,
        'avg_qQ': avg_qQ,
        'max_qQ': max_qQ,
        'avg_gpad': avg_gpad,
    },
    'basins': basins_summary,
    'diam_breakdown': diam_summary,
    'qclass_by_method': qclass_by_method,
    'depth_status': dict(depth_status_counts),
    'qQ_hist': {
        'bins': qQ_labels,
        'counts': qQ_hist,
    },
    'gpad_hist': {
        'bins': ['0–2.5K','2.5K–5K','5K–7.5K','7.5K–10K','10K–12.5K','12.5K–15K','15K–20K','20K–30K'],
        'counts': gpad_hist,
    },
    'scatter': scatter_sample,
    'mt_qclass': {mt: dict(c) for mt, c in mt_qclass.items()},
}

with open('capacity_sp_data.json', 'w') as f:
    json.dump(dashboard_data, f, indent=2)

print('Exported capacity_sp_data.json')
m = dashboard_data['meta']
print(f"  Surcharged: {m['n_surcharged']} ({m['surcharge_pct']}%)")
print(f"  q/Q def   : {m['n_qQ_deficient']} ({m['qQ_def_pct']}%)")
print(f"  Critical  : {m['n_critical']} ({m['critical_pct']}%)")
print(f"  Adequate  : {m['n_adequate']} ({m['adequate_pct']}%)")
print(f"  Basins    : {len(dashboard_data['basins'])}")
print(f"  Scatter   : {len(dashboard_data['scatter'])}")


Exported capacity_sp_data.json
  Surcharged: 105 (6.4%)
  q/Q def   : 1401 (85.7%)
  Critical  : 129 (7.9%)
  Adequate  : 1353 (82.8%)
  Basins    : 38
  Scatter   : 334
